# 🚀 COMPLETE PIPELINE: UML to Test Cases

## Three-Stage Architecture

```
PlantUML Files
      ↓
┌─────────────────────────────────────┐
│  STAGE 1: Parse & Correct Edges    │
│  - Extract actors, usecases, classes│
│  - Correct edge directions          │
│  - Learn action ordering            │
└─────────────────────────────────────┘
      ↓
┌─────────────────────────────────────┐
│  STAGE 2: Unified Graph Construction│
│  - Build NetworkX DiGraph           │
│  - O(V+E) complexity                │
│  - Add node/edge metadata           │
└─────────────────────────────────────┘
      ↓
┌─────────────────────────────────────┐
│  STAGE 3: Risk-Guided A* Paths     │
│  - GNN risk prediction              │
│  - Risk-Guided A* search            │
│  - Test path generation             │
│  - Coverage analysis                │
└─────────────────────────────────────┘
      ↓
   Test Paths
   (Ready for Stage 4: LLM Test Cases)
```

## 📦 Install Dependencies

In [ ]:
!pip install networkx matplotlib torch torch-geometric -q

## 📚 Import Stage 3 Module

Upload `stage3_risk_guided_astar.py` to your Colab environment, then import it:

In [ ]:
# Import all stages
import os
import re
import json
from collections import Counter
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Tuple, Optional

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Import Stage 3 (upload the .py file first)
from stage3_risk_guided_astar import (
    RiskPredictor,
    RiskGuidedAStar,
    TestPathGenerator,
    PathAnalyzer,
    export_paths_for_llm
)

---
# 🔵 STAGE 1: PlantUML Parser (Copy from previous notebook)

In [ ]:
# Include all Stage 1 functions from INTEGRATED_UML_TO_GRAPH.ipynb
# (parse_plantuml_raw, extract_action, build_action_order, etc.)

def parse_plantuml_raw(puml_path):
    """Parse PlantUML file and extract nodes and edges"""
    nodes = {}
    edges = []

    actor_pat   = re.compile(r'actor\s+"?(.+?)"?(\s+as\s+(\w+))?', re.I)
    usecase_pat = re.compile(r'usecase\s+"?(.+?)"?(\s+as\s+(\w+))?', re.I)
    class_pat   = re.compile(r'class\s+"?(.+?)"?(\s+as\s+(\w+))?', re.I)
    edge_pat    = re.compile(r'(.+?)\s*[-.]+[<>]*\s*(.+)')

    with open(puml_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("@"):
                continue

            m = actor_pat.match(line)
            if m:
                nodes[m.group(3) or m.group(1)] = "actor"
                continue

            m = usecase_pat.match(line)
            if m:
                nodes[m.group(3) or m.group(1)] = "usecase"
                continue

            m = class_pat.match(line)
            if m:
                nodes[m.group(3) or m.group(1)] = "class"
                continue

            m = edge_pat.match(line)
            if m:
                src = m.group(1).strip().strip('"')
                tgt = m.group(2).strip().strip('"')
                edges.append((src, tgt))
                nodes.setdefault(src, "usecase")
                nodes.setdefault(tgt, "usecase")

    return nodes, edges

def split_camelcase(text: str) -> list:
    """Split CamelCase into separate words"""
    spaced = re.sub(r'(?<!^)(?=[A-Z])', ' ', text)
    return spaced.split()

def extract_action(name: str) -> str:
    """Extract primary action from use case name"""
    words = split_camelcase(name)
    text = ' '.join(words)
    text = re.sub(r'[_\-]', ' ', text)
    tokens = re.findall(r"[a-zA-Z]+", text)
    
    if not tokens:
        return None
    
    action = tokens[0].lower()
    
    if len(action) < 3:
        return None
    if action.endswith(("service", "system", "manager")):
        return None
    if action in ("user", "customer", "patient", "admin", "manager", "team", "actor"):
        return None
    
    return action

def build_action_order(all_usecase_names):
    """Build frequency-based action ordering"""
    freq = Counter()
    for name in all_usecase_names:
        action = extract_action(name)
        if action:
            freq[action] += 1
    return [a for a, _ in freq.most_common()]

def rank(name, action_order):
    """Rank action by frequency"""
    name = name.lower()
    for i, action in enumerate(action_order):
        if action in name:
            return i
    return len(action_order)

def correct_edges(nodes, edges, action_order):
    """Correct edge directions based on node types and action ordering"""
    corrected = []

    for src, tgt in edges:
        s, t = nodes.get(src), nodes.get(tgt)

        if s == "actor" and t == "usecase":
            corrected.append((src, tgt))

        elif s == "usecase" and t == "actor":
            corrected.append((tgt, src))

        elif s == "usecase" and t == "usecase":
            if rank(src, action_order) <= rank(tgt, action_order):
                corrected.append((src, tgt))
            else:
                corrected.append((tgt, src))

        elif s == "usecase" and t == "class":
            corrected.append((src, tgt))

        elif s == "class" and t == "usecase":
            corrected.append((tgt, src))

    return corrected

def collect_all_usecases(dataset_root):
    """Collect all use cases from dataset"""
    all_usecases = []

    for root, _, files in os.walk(dataset_root):
        for file in files:
            if file.endswith(".puml"):
                nodes, _ = parse_plantuml_raw(os.path.join(root, file))
                for name, t in nodes.items():
                    if t == "usecase":
                        all_usecases.append(name)

    return all_usecases

def parse_and_correct_plantuml(puml_path, action_order):
    """Parse and correct a single PlantUML file"""
    nodes, raw_edges = parse_plantuml_raw(puml_path)
    edges = correct_edges(nodes, raw_edges, action_order)

    uml = {
        "actors": [],
        "usecases": [],
        "classes": [],
        "relations": []
    }

    type_map = {
        "actor": "actors",
        "usecase": "usecases",
        "class": "classes"
    }

    id_map = {}
    counter = 0

    def get_id(name):
        nonlocal counter
        if name not in id_map:
            id_map[name] = f"N{counter}"
            counter += 1
        return id_map[name]

    for name, t in nodes.items():
        if t in type_map:
            uml[type_map[t]].append((get_id(name), name))

    for src, tgt in edges:
        uml["relations"].append((get_id(src), get_id(tgt)))

    return uml

def parse_and_correct_dataset(dataset_root, action_order):
    """Parse entire dataset"""
    corrected_models = []

    for root, _, files in os.walk(dataset_root):
        for file in files:
            if file.endswith(".puml"):
                uml = parse_and_correct_plantuml(
                    os.path.join(root, file),
                    action_order
                )
                corrected_models.append(uml)

    return corrected_models
# Add other Stage 1 functions here...
# (See INTEGRATED_UML_TO_GRAPH.ipynb for full implementation)

---
# 🟢 STAGE 2: Unified Graph Construction (Copy from previous notebook)

In [ ]:
# Include all Stage 2 classes from INTEGRATED_UML_TO_GRAPH.ipynb
# (NodeType, EdgeType, UMLElements, UnifiedGraphBuilder)

class NodeType(Enum):
    """Types of nodes in the unified graph"""
    ACTOR = "actor"
    USECASE = "usecase"
    CLASS = "class"
    ACTIVITY = "activity"


class EdgeType(Enum):
    """Types of relationships/edges in the unified graph"""
    INCLUDE = "include"          # Use case includes another
    EXTEND = "extend"            # Use case extends another
    DEPENDENCY = "dependency"     # General dependency
    CONTROL_FLOW = "control_flow" # Sequential flow
    ASSOCIATION = "association"   # General association
    INVOKES = "invokes"          # Actor invokes use case
    USES = "uses"                # Use case uses class/system


@dataclass
class UMLElements:
    """Container for parsed UML elements"""
    actors: List[Tuple[str, str]] = field(default_factory=list)
    usecases: List[Tuple[str, str]] = field(default_factory=list)
    classes: List[Tuple[str, str]] = field(default_factory=list)
    activities: List[Tuple[str, str]] = field(default_factory=list)
    
    # Relationships with types
    includes: List[Tuple[str, str]] = field(default_factory=list)
    extends: List[Tuple[str, str]] = field(default_factory=list)
    dependencies: List[Tuple[str, str]] = field(default_factory=list)
    control_flows: List[Tuple[str, str]] = field(default_factory=list)
    associations: List[Tuple[str, str]] = field(default_factory=list)

print("✓ Data structures defined")
class UnifiedGraphBuilder:
    """
    Builds a unified behavioral graph from UML elements
    Complexity: O(V + E) where V = nodes, E = edges
    """
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.nodes = {}
        
    def build_unified_graph(self, uml_elements):
        """
        Main algorithm to build unified graph
        """
        # Step 1: Initialize directed graph G
        self.graph = nx.DiGraph()
        self.nodes.clear()
        
        print("Building Unified Graph...")
        print("-" * 60)
        
        # Step 2: Add class nodes
        print(f"Step 2: Adding {len(uml_elements.classes)} class nodes...")
        for class_id, class_name in uml_elements.classes:
            self._add_node(class_id, class_name, NodeType.CLASS)
        
        # Step 3: Add use case nodes
        print(f"Step 3: Adding {len(uml_elements.usecases)} use case nodes...")
        for uc_id, uc_name in uml_elements.usecases:
            self._add_node(uc_id, uc_name, NodeType.USECASE)
        
        # Step 4: Add activity nodes
        print(f"Step 4: Adding {len(uml_elements.activities)} activity nodes...")
        for act_id, act_name in uml_elements.activities:
            self._add_node(act_id, act_name, NodeType.ACTIVITY)
        
        # Step 4b: Add actor nodes
        print(f"Step 4b: Adding {len(uml_elements.actors)} actor nodes...")
        for actor_id, actor_name in uml_elements.actors:
            self._add_node(actor_id, actor_name, NodeType.ACTOR)
        
        # Step 5: Add relationships
        print(f"Step 5: Adding relationships...")
        
        print(f"  - {len(uml_elements.includes)} include edges")
        for src, tgt in uml_elements.includes:
            self._add_edge(src, tgt, EdgeType.INCLUDE)
        
        print(f"  - {len(uml_elements.extends)} extend edges")
        for src, tgt in uml_elements.extends:
            self._add_edge(src, tgt, EdgeType.EXTEND)
        
        print(f"  - {len(uml_elements.dependencies)} dependency edges")
        for src, tgt in uml_elements.dependencies:
            self._add_edge(src, tgt, EdgeType.DEPENDENCY)
        
        print(f"  - {len(uml_elements.control_flows)} control flow edges")
        for src, tgt in uml_elements.control_flows:
            self._add_edge(src, tgt, EdgeType.CONTROL_FLOW)
        
        print(f"  - {len(uml_elements.associations)} association edges")
        for src, tgt in uml_elements.associations:
            self._add_edge(src, tgt, EdgeType.ASSOCIATION)
        
        # Step 6: Return unified graph
        print("-" * 60)
        print(f"✓ Unified graph constructed:")
        print(f"  Total nodes (V): {self.graph.number_of_nodes()}")
        print(f"  Total edges (E): {self.graph.number_of_edges()}")
        print(f"  Complexity: O({self.graph.number_of_nodes()} + {self.graph.number_of_edges()}) = O({self.graph.number_of_nodes() + self.graph.number_of_edges()})")
        
        return self.graph
    
    def _add_node(self, node_id, name, node_type):
        """Add a node to the unified graph"""
        if node_id not in self.nodes:
            self.nodes[node_id] = {
                'name': name,
                'type': node_type.value
            }
            self.graph.add_node(
                node_id,
                name=name,
                node_type=node_type.value,
                label=name
            )
    
    def _add_edge(self, source, target, edge_type):
        """Add an edge to the unified graph"""
        if source in self.nodes and target in self.nodes:
            self.graph.add_edge(
                source,
                target,
                edge_type=edge_type.value,
                label=edge_type.value
            )
    
    def get_statistics(self):
        """Get statistics about the unified graph"""
        stats = {
            'total_nodes': self.graph.number_of_nodes(),
            'total_edges': self.graph.number_of_edges(),
            'node_types': {},
            'edge_types': {},
            'density': nx.density(self.graph),
            'is_dag': nx.is_directed_acyclic_graph(self.graph)
        }
        
        # Count nodes by type
        for node_id, data in self.graph.nodes(data=True):
            node_type = data.get('node_type', 'unknown')
            stats['node_types'][node_type] = stats['node_types'].get(node_type, 0) + 1
        
        # Count edges by type
        for src, tgt, data in self.graph.edges(data=True):
            edge_type = data.get('edge_type', 'unknown')
            stats['edge_types'][edge_type] = stats['edge_types'].get(edge_type, 0) + 1
        
        return stats

print("✓ UnifiedGraphBuilder class defined")
# Add other Stage 2 classes here...

def visualize_unified_graph(graph: nx.DiGraph, title="Unified Behavioral Graph G = (V, E)"):
    """Visualize the unified graph"""
    plt.figure(figsize=(14, 10))
    
    # Create layout
    pos = nx.spring_layout(graph, k=2, iterations=50, seed=42)
    
    # Color nodes by type
    node_colors = []
    color_map = {
        'actor': '#FF6B6B',
        'usecase': '#4ECDC4',
        'class': '#45B7D1',
        'activity': '#FFA07A'
    }
    
    for node in graph.nodes():
        node_type = graph.nodes[node].get('node_type', 'unknown')
        node_colors.append(color_map.get(node_type, '#95A5A6'))
    
    # Draw nodes
    nx.draw_networkx_nodes(
        graph, pos,
        node_color=node_colors,
        node_size=1200,
        alpha=0.9
    )
    
    # Draw edges
    edge_colors = []
    for src, tgt in graph.edges():
        edge_type = graph[src][tgt].get('edge_type', 'unknown')
        
        if edge_type == 'include':
            edge_colors.append('#2ECC71')
        elif edge_type == 'extend':
            edge_colors.append('#E74C3C')
        elif edge_type == 'control_flow':
            edge_colors.append('#3498DB')
        else:
            edge_colors.append('#95A5A6')
    
    nx.draw_networkx_edges(
        graph, pos,
        edge_color=edge_colors,
        width=2.5,
        alpha=0.6,
        arrows=True,
        arrowsize=20,
        arrowstyle='->'
    )
    
    # Draw labels
    labels = {node: graph.nodes[node].get('name', node) for node in graph.nodes()}
    nx.draw_networkx_labels(
        graph, pos,
        labels,
        font_size=9,
        font_weight='bold'
    )
    
    # Create legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#FF6B6B', markersize=12, label='Actor'),
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#4ECDC4', markersize=12, label='UseCase'),
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#45B7D1', markersize=12, label='Class'),
        Line2D([0], [0], marker='o', color='w', 
              markerfacecolor='#FFA07A', markersize=12, label='Activity'),
        Line2D([0], [0], color='#2ECC71', linewidth=2, label='Include'),
        Line2D([0], [0], color='#E74C3C', linewidth=2, label='Extend'),
        Line2D([0], [0], color='#3498DB', linewidth=2, label='Control Flow'),
    ]
    plt.legend(handles=legend_elements, loc='upper left', 
              bbox_to_anchor=(1, 1), fontsize=10)
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("✓ Visualization function defined")

# Create sample UML elements
uml_elements = UMLElements(
    actors=[
        ('A1', 'User'),
        ('A2', 'Admin')
    ],
    usecases=[
        ('UC1', 'Login'),
        ('UC2', 'ValidateCredentials'),
        ('UC3', 'CheckPermissions'),
        ('UC4', 'AccessSystem'),
        ('UC5', 'ManageUsers')
    ],
    classes=[
        ('C1', 'AuthService'),
        ('C2', 'UserDatabase'),
        ('C3', 'PermissionManager')
    ],
    activities=[
        ('ACT1', 'EnterCredentials'),
        ('ACT2', 'SubmitForm'),
        ('ACT3', 'ProcessRequest')
    ]
)

# Define relationships
uml_elements.includes = [
    ('UC1', 'UC2'),  # Login includes ValidateCredentials
    ('UC4', 'UC3'),  # AccessSystem includes CheckPermissions
]

uml_elements.extends = [
    ('UC1', 'UC5'),  # Login extends to ManageUsers (for admin)
]

uml_elements.dependencies = [
    ('UC2', 'C1'),  # ValidateCredentials depends on AuthService
    ('UC2', 'C2'),  # ValidateCredentials depends on UserDatabase
    ('UC3', 'C3'),  # CheckPermissions depends on PermissionManager
]

uml_elements.control_flows = [
    ('ACT1', 'ACT2'),  # EnterCredentials → SubmitForm
    ('ACT2', 'ACT3'),  # SubmitForm → ProcessRequest
]

uml_elements.associations = [
    ('A1', 'UC1'),  # User invokes Login
    ('A2', 'UC5'),  # Admin invokes ManageUsers
    ('UC1', 'UC4'),  # Login leads to AccessSystem
]

print("✓ Sample UML elements created")
# (See INTEGRATED_UML_TO_GRAPH.ipynb for full implementation)

In [ ]:
# Build unified graph
builder = UnifiedGraphBuilder()
unified_graph = builder.build_unified_graph(uml_elements)

In [ ]:
# Get statistics
stats = builder.get_statistics()

print("\n" + "="*60)
print("UNIFIED GRAPH STATISTICS")
print("="*60)
print(f"\nGraph Structure:")
print(f"  G = (V, E)")
print(f"  |V| = {stats['total_nodes']} nodes")
print(f"  |E| = {stats['total_edges']} edges")
print(f"\nGraph Properties:")
print(f"  Density: {stats['density']:.4f}")
print(f"  Is DAG: {stats['is_dag']}")
print(f"\nNode Distribution:")
for node_type, count in stats['node_types'].items():
    print(f"  {node_type}: {count}")
print(f"\nEdge Distribution:")
for edge_type, count in stats['edge_types'].items():
    print(f"  {edge_type}: {count}")
print(f"\nComplexity Analysis:")
print(f"  Time: O(V + E) = O({stats['total_nodes']} + {stats['total_edges']}) = O({stats['total_nodes'] + stats['total_edges']})")
print(f"  Space: O(V + E) = O({stats['total_nodes'] + stats['total_edges']})")
print("="*60)

In [ ]:
# Visualize the unified graph
visualize_unified_graph(unified_graph)

---
# 🔴 STAGE 3: Risk-Guided A* Path Generation

## NEW: Test Path Generation with Risk Analysis

## Example 1: Generate Test Paths from Single UML Diagram

In [ ]:
# Step 1: Parse PlantUML (Stage 1)
PUML_FILE = "ifa_football/ifa_class_diagram.puml"
DATASET_PATH="ifa_football"
all_usecases = collect_all_usecases(DATASET_PATH)
action_order = build_action_order(all_usecases)

uml_dict = parse_and_correct_plantuml(PUML_FILE, action_order)

# Step 2: Build Unified Graph (Stage 2)
uml_elements = UMLElements(
    actors=uml_dict['actors'],
    usecases=uml_dict['usecases'],
    classes=uml_dict['classes'],
    associations=uml_dict['relations']
)

builder = UnifiedGraphBuilder()
graph = builder.build_unified_graph(uml_elements)

print(f"Graph built: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# Step 3: Generate Test Paths (Stage 3 - NEW!)
path_generator = TestPathGenerator(graph)

test_paths = path_generator.generate_test_paths(
    num_paths=10,
    max_path_length=15,
    remove_redundant=True,
    redundancy_threshold=0.8
)

# Print summary
path_generator.print_path_summary()

## Example 2: Visualize Generated Paths

In [ ]:
# Visualize top 3 paths
for i in range(min(3, len(test_paths))):
    path_generator.visualize_path(i, output_path=f'test_path_{i+1}.png')

## Example 3: Export for LLM Test Case Generation (Stage 4)

In [ ]:
# Export paths in format ready for LLM
export_data = export_paths_for_llm(
    test_paths,
    graph,
    output_path='paths_for_llm.json'
)

print("\nExported data structure:")
print(f"  Paths: {len(export_data['paths'])}")
print(f"  Nodes per path (avg): {np.mean([len(p['nodes']) for p in export_data['paths']]):.1f}")
print(f"  Average risk: {np.mean([p['risk_score'] for p in export_data['paths']]):.3f}")

## Example 4: Batch Process Multiple Diagrams

In [ ]:
# Process entire dataset
DATASET_PATH = "ifa_football"

all_test_paths = []
all_graphs = []

# Learn action order once
all_usecases = collect_all_usecases(DATASET_PATH)
action_order = build_action_order(all_usecases)

# Process each file
for root, _, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".puml"):
            filepath = os.path.join(root, file)
            
            print(f"\nProcessing: {file}")
            
            # Stage 1 & 2
            uml_dict = parse_and_correct_plantuml(filepath, action_order)
            uml_elements = UMLElements(
                actors=uml_dict['actors'],
                usecases=uml_dict['usecases'],
                classes=uml_dict['classes'],
                associations=uml_dict['relations']
            )
            
            builder = UnifiedGraphBuilder()
            graph = builder.build_unified_graph(uml_elements)
            all_graphs.append(graph)
            
            # Stage 3
            path_gen = TestPathGenerator(graph)
            paths = path_gen.generate_test_paths(num_paths=5)
            all_test_paths.extend(paths)
            
            print(f"  Generated {len(paths)} paths")

print(f"\n{'='*70}")
print(f"DATASET PROCESSING COMPLETE")
print(f"{'='*70}")
print(f"Total graphs: {len(all_graphs)}")
print(f"Total test paths: {len(all_test_paths)}")
print(f"Average paths per diagram: {len(all_test_paths) / len(all_graphs):.1f}")

---
# 📊 ANALYSIS & METRICS

## Coverage Analysis

In [ ]:
# Analyze coverage
coverage_report = path_generator.get_coverage_report()

print("="*70)
print("COVERAGE ANALYSIS")
print("="*70)

print(f"\nOverall Coverage:")
print(f"  Node Coverage: {coverage_report['node_coverage']*100:.1f}%")
print(f"  Edge Coverage: {coverage_report['edge_coverage']*100:.1f}%")

print(f"\nCoverage by Node Type:")
for node_type, stats in coverage_report['type_coverage'].items():
    print(f"  {node_type:10}: {stats['coverage']*100:5.1f}% ({stats['covered']}/{stats['total']})")

## Risk Distribution Analysis

In [ ]:
# Analyze risk distribution
risk_scores_list = [path.risk_score for path in test_paths]

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(risk_scores_list, bins=20, color='#FF6B6B', alpha=0.7, edgecolor='black')
plt.xlabel('Risk Score', fontsize=12)
plt.ylabel('Number of Paths', fontsize=12)
plt.title('Risk Score Distribution', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Box plot
plt.subplot(1, 2, 2)
plt.boxplot(risk_scores_list, vert=True)
plt.ylabel('Risk Score', fontsize=12)
plt.title('Risk Score Statistics', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('risk_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nRisk Score Statistics:")
print(f"  Mean: {np.mean(risk_scores_list):.3f}")
print(f"  Median: {np.median(risk_scores_list):.3f}")
print(f"  Std Dev: {np.std(risk_scores_list):.3f}")
print(f"  Min: {np.min(risk_scores_list):.3f}")
print(f"  Max: {np.max(risk_scores_list):.3f}")

## Path Length vs Risk Analysis

In [ ]:
# Analyze relationship between path length and risk
lengths = [len(path.nodes) for path in test_paths]
risks = [path.risk_score for path in test_paths]

plt.figure(figsize=(10, 6))
plt.scatter(lengths, risks, s=100, alpha=0.6, c=risks, cmap='RdYlGn_r', edgecolors='black')
plt.colorbar(label='Risk Score')
plt.xlabel('Path Length (number of nodes)', fontsize=12)
plt.ylabel('Risk Score', fontsize=12)
plt.title('Path Length vs Risk Score', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('length_vs_risk.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation
correlation = np.corrcoef(lengths, risks)[0, 1]
print(f"\nCorrelation (length vs risk): {correlation:.3f}")

---
# 🎯 COMPLETE PIPELINE FUNCTION

In [ ]:
def uml_to_test_paths(puml_file: str,
                      dataset_path: str,
                      num_paths: int = 10,
                      visualize: bool = True) -> List:
    """
    Complete pipeline: PlantUML → Test Paths
    
    Args:
        puml_file: Path to PlantUML file
        dataset_path: Dataset path for action order learning
        num_paths: Number of test paths to generate
        visualize: Whether to visualize results
        
    Returns:
        List of TestPath objects
    """
    print("\n" + "="*70)
    print("COMPLETE PIPELINE: UML TO TEST PATHS")
    print("="*70)
    
    # STAGE 1: Parse PlantUML
    print("\n[STAGE 1] Parsing PlantUML...")
    all_usecases = collect_all_usecases(dataset_path)
    action_order = build_action_order(all_usecases)
    uml_dict = parse_and_correct_plantuml(puml_file, action_order)
    print(f"  ✓ Parsed: {len(uml_dict['actors'])} actors, {len(uml_dict['usecases'])} usecases")
    
    # STAGE 2: Build Unified Graph
    print("\n[STAGE 2] Building unified graph...")
    uml_elements = UMLElements(
        actors=uml_dict['actors'],
        usecases=uml_dict['usecases'],
        classes=uml_dict['classes'],
        associations=uml_dict['relations']
    )
    builder = UnifiedGraphBuilder()
    graph = builder.build_unified_graph(uml_elements)
    print(f"  ✓ Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
    
    # STAGE 3: Generate Test Paths
    print("\n[STAGE 3] Generating test paths...")
    path_gen = TestPathGenerator(graph)
    test_paths = path_gen.generate_test_paths(num_paths=num_paths)
    
    # Print results
    path_gen.print_path_summary()
    
    # Visualize if requested
    if visualize and test_paths:
        print("\n[VISUALIZATION] Creating visualizations...")
        path_gen.visualize_path(0, 'top_risk_path.png')
    
    # Export for Stage 4
    export_paths_for_llm(test_paths, graph, 'test_paths_export.json')
    
    print("\n" + "="*70)
    print("PIPELINE COMPLETE!")
    print("="*70)
    
    return test_paths


# Usage
test_paths = uml_to_test_paths(
    puml_file="ifa_football/audit_budgets_001.puml",
    dataset_path="ifa_football",
    num_paths=10,
    visualize=True
)

---
# 📤 EXPORT FOR STAGE 4 (LLM TEST CASE GENERATION)

In [ ]:
# Final export for LLM
final_export = export_paths_for_llm(
    test_paths,
    graph,
    output_path='final_test_paths.json'
)

# Preview export
print("\nExport Preview (First Path):")
print(json.dumps(final_export['paths'][0], indent=2))

---
# ✅ SUMMARY

## What We've Built:

### **Stage 1: PlantUML Parser** ✓
- Extracts actors, use cases, classes
- Corrects edge directions
- Learns action ordering

### **Stage 2: Unified Graph Construction** ✓
- Builds NetworkX directed graph
- O(V+E) complexity
- Maintains UML semantics

### **Stage 3: Risk-Guided A* Path Generation** ✓ NEW!
- GNN-based risk prediction
- Risk-guided A* search algorithm
- Test path generation with priority
- Coverage analysis
- Redundancy elimination

## Key Features:

1. **Risk Prediction**: GNN learns structural patterns and predicts risk
2. **Intelligent Search**: A* prioritizes high-risk nodes
3. **Path Optimization**: Removes redundant paths
4. **Coverage Analysis**: Tracks node/edge coverage
5. **Priority Ranking**: Assigns priority to test paths
6. **Visualization**: Visual representation of paths
7. **Export Ready**: JSON format for Stage 4 (LLM)

## Next Step: Stage 4

**LLM-Based Test Case Generation**
- Convert paths to natural language test cases
- Generate test case variations
- Add preconditions, steps, expected results

## Parameters from PPT:

✓ Number of UML nodes  
✓ Number of UML relationships  
✓ Node risk score (GNN output)  
✓ Path cost (A* algorithm)  
✓ Heuristic function value  
✓ Coverage percentage  
✓ Test case complexity level  
✓ Redundancy threshold  
✓ Priority weight factor  

🎉 **Complete Three-Stage Pipeline Ready!** 🎉